# 10 — Attention Visualization
Visualize learned attention coefficients when available from GAT layers.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

PROJECT_ROOT = Path("..").resolve()
edges = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "segment_graph_edges.csv")
mapping = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "road_segment_mapping.csv")


In [ ]:
try:
    from torch_geometric.nn import GATConv
    class AttnProbe(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.conv = GATConv(4, 8, heads=2, dropout=0.0)
        def forward(self, x, edge_index):
            out, (ei, alpha) = self.conv(x, edge_index, return_attention_weights=True)
            return out, ei, alpha

    model = AttnProbe()
    x = torch.randn(len(mapping), 4)
    edge_index = torch.tensor(edges[["src","dst"]].to_numpy().T, dtype=torch.long)
    _, ei, alpha = model(x, edge_index)
    alpha_mean = alpha.mean(dim=1).detach().numpy()

    plt.figure(figsize=(6,3))
    plt.hist(alpha_mean, bins=40)
    plt.title("Attention coefficient distribution")
    plt.xlabel("alpha")
    plt.ylabel("count")
    plt.tight_layout(); plt.show()

    top_idx = np.argsort(-alpha_mean)[:10]
    top_edges = pd.DataFrame({"src": ei[0][top_idx].numpy(), "dst": ei[1][top_idx].numpy(), "alpha": alpha_mean[top_idx]})
    print(top_edges)
except Exception as e:
    print("Attention extraction unavailable in current setup:", e)


## Caution
Attention is a model-internal weighting signal, not guaranteed causal evidence.


In [ ]:
print("Compare attention patterns across scenario-conditioned inputs in future iterations.")
